[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C64_ML_Knowledge_QA_Course/01_ml_basics/01_ml_basics_qa.ipynb)

# 01 · 机器学习基础问答（偏差-方差与双下降 / L1-L2稀疏性 / dropout集成等价性 / CV时序泄漏 / bagging-boosting实测）

目标：把模块 00 的"三段式答法"应用到 C07 已经推导过的六个 ML 基础主题上——
这里**不重复推导**，只用可运行的数值实验，把每个主题"什么时候会失效"这条最难背出来的边界，
变成你亲手跑出来、亲眼看到的结果。

本 notebook 你会亲手实现：
1. **偏差-方差数值分解** —— 复现经典 U 型曲线
2. **双下降** —— 用随机傅里叶特征 + 最小范数解，复现"插值阈值附近误差骤增、之后又下降"的现象
3. **L1 vs L2 的稀疏性** —— 正交设计下的闭式解，量化"L1精确清零、L2只收缩"
4. **dropout 作为集成的等价性** —— 线性 readout 下精确成立，非线性 readout 下只是近似
5. **CV 在时序数据上的泄漏** —— 量化 shuffled K-fold 比 walk-forward 乐观多少倍
6. **bagging 降方差 / boosting 降偏差** —— 在同一个玩具问题上实测两者的偏差-方差变化

> 心智模型：**C07 教你怎么推导这些结论；这里教你怎么亲手验证"它什么时候会不成立"。**

## 0 · 环境自检

In [ ]:
import sys
import numpy as np

print('Python :', sys.version.split()[0])
print('numpy  :', np.__version__)
assert sys.version_info >= (3, 8)

rng_check = np.random.default_rng(0)
assert rng_check.uniform(0, 1) is not None
print('\n✅ 环境自检通过：本课全程只用 numpy + 标准库，CPU 可跑，不联网。')

## 1 · 偏差-方差数值分解：复现经典 U 型曲线

真实函数 $f(x)=\sin(2x)+0.4x$ 加噪声，用不同阶数的多项式拟合。对每个阶数重复 300 次独立抽样训练集，
统计预测的均值（→ bias）和方差（→ variance）。

In [ ]:
rng = np.random.default_rng(0)

def true_fn(x):
    return np.sin(2.0 * x) + 0.4 * x

sigma = 0.25
n_train = 30
x_test = np.linspace(-1, 1, 50)
y_test_true = true_fn(x_test)

degs = [1, 2, 3, 5, 7, 9]
B = 300
result = {}
for deg in degs:
    preds = np.zeros((B, len(x_test)))
    for b in range(B):
        x_tr = rng.uniform(-1, 1, n_train)
        y_tr = true_fn(x_tr) + rng.normal(0, sigma, n_train)
        preds[b] = np.polyval(np.polyfit(x_tr, y_tr, deg), x_test)
    mean_pred = preds.mean(axis=0)
    bias2 = np.mean((mean_pred - y_test_true) ** 2)
    var = np.mean(preds.var(axis=0))
    result[deg] = (bias2, var, bias2 + var + sigma ** 2)

print(f'{"deg":>4} {"bias2":>10} {"var":>10} {"total":>10}')
for deg in degs:
    b2, v, t = result[deg]
    print(f'{deg:>4} {b2:>10.4f} {v:>10.4f} {t:>10.4f}')

# 断言：低阶到中阶，bias 应该显著下降；高阶时方差应该远超低阶（经典 U 型的"过拟合"一侧）
assert result[1][0] > result[3][0] * 50          # bias: deg1 远高于 deg3
assert result[9][1] > result[3][1] * 100         # var:  deg9 远高于 deg3（未加正则的高阶多项式方差爆炸）
assert result[9][2] > result[1][2] * 5           # 总误差：deg9 远超 deg1，说明"过拟合"确实更差
best_deg = min(degs, key=lambda d: result[d][2])
assert best_deg == 3
print(f'\n✅ U 型验证通过：最优阶数在 deg={best_deg}（bias 和 var 的总和最小）；'
      f'deg=9 时方差已经比 deg=3 大 100 倍以上——未加正则的高阶多项式基是数值上极不稳定的经典陷阱。')

## 2 · 双下降：随机傅里叶特征 + 最小范数解

真实关系是 5 维输入上的线性函数。用 $P$ 个随机傅里叶特征 $z=\cos(Xw+b)$ 拟合，$P$ 从远小于训练样本数
一路增大到远大于训练样本数，$P$ 每次都取**最小范数**解（`np.linalg.pinv`）。

关键现象：误差在 $P\approx n_{\text{train}}$（插值阈值）附近骤增，之后随 $P$ 继续增大反而下降——
这正是双下降，也是模块 01 第 1 节"什么时候失效"的实证。

In [ ]:
rng2 = np.random.default_rng(1)

d_true, n_train2, n_test = 5, 40, 200
w_star = rng2.normal(size=d_true)
sigma2 = 0.5

X_train = rng2.normal(size=(n_train2, d_true))
y_train = X_train @ w_star + rng2.normal(0, sigma2, n_train2)
X_test = rng2.normal(size=(n_test, d_true))
y_test = X_test @ w_star + rng2.normal(0, sigma2, n_test)

Ps = [5, 20, 35, 40, 42, 50, 80, 200, 800]
P_max = max(Ps)
W_full = rng2.normal(size=(d_true, P_max))
b_full = rng2.uniform(0, 2 * np.pi, P_max)

def rff_features(X, P):
    return np.cos(X @ W_full[:, :P] + b_full[:P])

mse = {}
for P in Ps:
    Z_train = rff_features(X_train, P)
    Z_test = rff_features(X_test, P)
    beta = np.linalg.pinv(Z_train) @ y_train      # 最小范数解，欠定/超定都适用
    pred = Z_test @ beta
    mse[P] = np.mean((pred - y_test) ** 2)

print(f'{"P":>5} {"test_mse":>12}')
for P in Ps:
    print(f'{P:>5} {mse[P]:>12.4f}')

peak = max(mse.values())
assert mse[40] == peak                              # 插值阈值(P≈n_train=40)附近误差骤增到最大
assert mse[40] > mse[35] and mse[40] > mse[42]       # 峰值两侧都更低——"骤增"是局部的
assert mse[800] < mse[40] / 50                       # 过参数化到 P=800 时，误差比峰值低 50 倍以上
assert mse[800] < mse[35]                            # 甚至比"峰值之前"的欠参数化区间还低——这才是双下降的关键
print(f'\n✅ 双下降复现：峰值在 P={list(mse.keys())[list(mse.values()).index(peak)]}（插值阈值附近），'
      f'P=800 时误差 {mse[800]:.3f} 反而低于 P=35 时的 {mse[35]:.3f}。')
print('   这个现象无法用"方差随复杂度单调增加"的经典图像解释——过参数化区间的最小范数解自带隐式正则。')

## 3 · L1 vs L2 的稀疏性：正交设计下的闭式解

完整 KKT 推导见 C07-06；这里只验证结论本身。在正交设计（$X^TX=I$）下，Ridge 和 Lasso 都有闭式解：
$\hat\beta_{\text{ridge}}=\hat\beta_{\text{ols}}/(1+\lambda)$，
$\hat\beta_{\text{lasso}}=\text{sign}(\hat\beta_{\text{ols}})\cdot\max(|\hat\beta_{\text{ols}}|-\lambda,0)$（软阈值）。

In [ ]:
beta_ols = np.array([3.0, -0.5, 1.2, 0.05, -2.0])

def ridge_shrink(beta, lam):
    return beta / (1 + lam)

def lasso_shrink(beta, lam):
    return np.sign(beta) * np.maximum(np.abs(beta) - lam, 0.0)

lam = 1.0
r = ridge_shrink(beta_ols, lam)
l = lasso_shrink(beta_ols, lam)
print('OLS  :', beta_ols)
print('Ridge:', r)
print('Lasso:', l)

n_zero_ridge = np.sum(np.isclose(r, 0.0))
n_zero_lasso = np.sum(np.isclose(l, 0.0))
assert n_zero_ridge == 0
assert n_zero_lasso == 2                         # |-0.5| 和 |0.05| 都 <= lam=1.0，被精确压成 0
assert np.allclose(l, [2.0, 0.0, 0.2, 0.0, -1.0])

# 解路径：随 lambda 增大，lasso 非零系数个数单调不增
lambdas = [0.0, 0.3, 0.6, 1.0, 2.0, 3.5]
nnz = [int(np.sum(~np.isclose(lasso_shrink(beta_ols, lam), 0.0))) for lam in lambdas]
print('\nlambda            :', lambdas)
print('lasso非零系数个数  :', nnz)
assert nnz == sorted(nnz, reverse=True)
print('\n✅ 正交设计下：L1 随 lambda 增大产生越来越多精确 0；L2 只收缩，从不精确为 0。')

## 4 · dropout 作为集成：线性 readout 下精确成立，非线性下只是近似

对输入做 Bernoulli(p) 的 dropout mask，比较"大量随机 mask 取平均"和"直接用权重缩放(×p)"两种做法。
线性输出层下二者在期望上完全相等；非线性 readout（sigmoid）下二者存在系统性的 Jensen gap，不会随采样数增加而消失。

In [ ]:
rng3 = np.random.default_rng(2)
d, p = 6, 0.7
x = rng3.normal(size=d)
w = rng3.normal(size=d)

exact_linear = p * (w @ x)

def mc_dropout_linear(n_samples):
    masks = (rng3.random((n_samples, d)) < p).astype(float)
    return ((masks * x) @ w).mean()

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def mc_dropout_nonlinear(n_samples):
    masks = (rng3.random((n_samples, d)) < p).astype(float)
    z = (masks * x) @ w
    return sigmoid(z).mean()

approx_nonlinear = sigmoid(p * (w @ x))

print(f'{"n":>8} {"MC(线性)":>10} {"解析值":>10} {"|diff|":>8} | {"MC(sigmoid)":>12} {"近似值":>10} {"gap":>8}')
for n in [200, 2000, 20000, 200000]:
    el = mc_dropout_linear(n)
    en = mc_dropout_nonlinear(n)
    print(f'{n:>8} {el:>10.5f} {exact_linear:>10.5f} {abs(el-exact_linear):>8.5f} | '
          f'{en:>12.5f} {approx_nonlinear:>10.5f} {abs(en-approx_nonlinear):>8.5f}')

final_linear = mc_dropout_linear(200000)
final_nonlinear = mc_dropout_nonlinear(200000)
assert abs(final_linear - exact_linear) < 0.01        # 线性readout：MC均值收敛到解析的权重缩放值
assert abs(final_nonlinear - approx_nonlinear) > 0.02  # 非线性readout：gap持续存在，不随采样数消失
print('\n✅ 验证通过：dropout≈权重缩放这条等价性，只在"下一步是线性组合"时精确成立；'
      '一旦中间插入非线性（如本例的sigmoid），就只是近似，而且这个 gap 不会随蒙特卡洛采样数增加而消失'
      '（Jensen不等式决定的系统性偏差，不是采样噪声）。')

## 5 · CV 在时序数据上的泄漏：量化 shuffled K-fold 比 walk-forward 乐观多少倍

构造一条随机游走序列（自相关极强、且未来增量独立于过去、本质不可预测）。
用 1-NN（按时间索引找最近邻）分别在 shuffled K-fold 和 walk-forward 两种切分下评估，比较 MSE。

In [ ]:
rng4 = np.random.default_rng(3)
N = 300
y_walk = np.cumsum(rng4.normal(0, 1.0, N))
t_idx = np.arange(N).reshape(-1, 1).astype(float)

def nn_predict(t_train, y_train, t_query):
    preds = np.empty(len(t_query))
    for i, tq in enumerate(t_query):
        j = np.argmin(np.abs(t_train[:, 0] - tq[0]))
        preds[i] = y_train[j]
    return preds

# 方案 A：标准 shuffled K-fold（错误做法——test点的"未来"近邻很可能落在训练集里）
K = 5
idx = rng4.permutation(N)
folds = np.array_split(idx, K)
mse_shuffled = []
for k in range(K):
    test_idx = folds[k]
    train_idx = np.concatenate([folds[j] for j in range(K) if j != k])
    pred = nn_predict(t_idx[train_idx], y_walk[train_idx], t_idx[test_idx])
    mse_shuffled.append(np.mean((pred - y_walk[test_idx]) ** 2))
mse_shuffled = float(np.mean(mse_shuffled))

# 方案 B：walk-forward（正确做法——只用过去预测未来）
splits = [(150, 180), (180, 210), (210, 240), (240, 270), (270, 300)]
mse_forward = []
for tr_end, te_end in splits:
    train_idx = np.arange(0, tr_end)
    test_idx = np.arange(tr_end, te_end)
    pred = nn_predict(t_idx[train_idx], y_walk[train_idx], t_idx[test_idx])
    mse_forward.append(np.mean((pred - y_walk[test_idx]) ** 2))
mse_forward = float(np.mean(mse_forward))

print(f'shuffled K-fold MSE = {mse_shuffled:.4f}')
print(f'walk-forward   MSE = {mse_forward:.4f}')
print(f'比值 walk-forward / shuffled = {mse_forward/mse_shuffled:.2f}x')

assert mse_forward > 5 * mse_shuffled     # 真实差距约20倍，这里用5倍做保守断言
print('\n✅ 验证通过：在自相关数据上，标准K折会让"未来"信息通过近邻泄漏进训练集，'
      '把真实误差低估一个数量级以上。')

## 6 · bagging 降方差 / boosting 降偏差：同一个玩具问题上的实测

真实函数 $f(x)=\sin(3x)$。bagging 用 1-NN（低偏差高方差）做基学习器；boosting 用深度1回归树桩
（高偏差低方差）做基学习器，通过拟合残差逐轮降低偏差。

In [ ]:
rng5 = np.random.default_rng(4)

def true_fn2(x):
    return np.sin(3.0 * x)

sigma5 = 0.3
n_train5 = 40
x_test5 = np.linspace(-2, 2, 40)
y_test5_true = true_fn2(x_test5)

def nn_predict_1d(x_train, y_train, x_query):
    preds = np.empty(len(x_query))
    for i, xq in enumerate(x_query):
        j = np.argmin(np.abs(x_train - xq))
        preds[i] = y_train[j]
    return preds

R, B = 200, 25
single_preds = np.zeros((R, len(x_test5)))
bagged_preds = np.zeros((R, len(x_test5)))
for r in range(R):
    x_tr = rng5.uniform(-2, 2, n_train5)
    y_tr = true_fn2(x_tr) + rng5.normal(0, sigma5, n_train5)
    single_preds[r] = nn_predict_1d(x_tr, y_tr, x_test5)
    boot_preds = np.zeros((B, len(x_test5)))
    for b in range(B):
        bidx = rng5.integers(0, n_train5, n_train5)
        boot_preds[b] = nn_predict_1d(x_tr[bidx], y_tr[bidx], x_test5)
    bagged_preds[r] = boot_preds.mean(axis=0)

def bias2_var(preds):
    mp = preds.mean(axis=0)
    return np.mean((mp - y_test5_true) ** 2), np.mean(preds.var(axis=0))

b2_single, v_single = bias2_var(single_preds)
b2_bag, v_bag = bias2_var(bagged_preds)
print(f'1-NN 单模型  : bias2={b2_single:.4f}  var={v_single:.4f}')
print(f'1-NN bagging : bias2={b2_bag:.4f}  var={v_bag:.4f}')
print(f'方差降低倍数 : {v_single/v_bag:.2f}x')

assert v_single > 1.3 * v_bag             # bagging 显著降方差
assert b2_single < 0.02 and b2_bag < 0.02 # bagging 几乎不改变（很低的）偏差

def fit_stump(x, r):
    order = np.argsort(x)
    xs, rs = x[order], r[order]
    best_sse, best_thr, best_l, best_rgt = np.inf, None, np.mean(rs), np.mean(rs)
    for i in range(1, len(xs)):
        if xs[i] == xs[i-1]:
            continue
        thr = (xs[i] + xs[i-1]) / 2
        left, right = rs[:i], rs[i:]
        lval, rval = left.mean(), right.mean()
        sse = np.sum((left - lval)**2) + np.sum((right - rval)**2)
        if sse < best_sse:
            best_sse, best_thr, best_l, best_rgt = sse, thr, lval, rval
    return best_thr, best_l, best_rgt

def stump_predict(thr, lval, rval, xq):
    if thr is None:
        return np.full(len(xq), lval)
    return np.where(xq < thr, lval, rval)

def boosted_predict(x_tr, y_tr, x_query, M, eta=0.5):
    F0 = np.mean(y_tr)
    stumps, residual = [], y_tr - F0
    for _ in range(M):
        thr, lval, rval = fit_stump(x_tr, residual)
        stumps.append((thr, lval, rval))
        residual = residual - eta * stump_predict(thr, lval, rval, x_tr)
    pred = np.full(len(x_query), F0)
    for thr, lval, rval in stumps:
        pred = pred + eta * stump_predict(thr, lval, rval, x_query)
    return pred

Ms = [1, 3, 6, 12, 25, 50]
boost_result = {}
for M in Ms:
    preds = np.zeros((R, len(x_test5)))
    for r in range(R):
        x_tr = rng5.uniform(-2, 2, n_train5)
        y_tr = true_fn2(x_tr) + rng5.normal(0, sigma5, n_train5)
        preds[r] = boosted_predict(x_tr, y_tr, x_test5, M)
    boost_result[M] = bias2_var(preds)

print(f'\n{"M":>4} {"bias2":>10} {"var":>10}')
for M in Ms:
    b2, v = boost_result[M]
    print(f'{M:>4} {b2:>10.4f} {v:>10.4f}')

bias_seq = [boost_result[M][0] for M in Ms]
var_seq = [boost_result[M][1] for M in Ms]
assert bias_seq == sorted(bias_seq, reverse=True)      # 偏差随轮数单调下降
assert bias_seq[-1] < bias_seq[0] / 10                 # 50轮后偏差降到1轮时的1/10以下
assert var_seq[-1] < var_seq[0] * 3                     # 方差只是缓慢增长（远没有偏差降得那么剧烈）
print('\n✅ 验证通过：bagging 主要压缩方差、几乎不动偏差；boosting 随轮数持续压缩偏差、方差缓慢上升。')

## ✏️ 练习 1：Lasso 软阈值算子

实现 `lasso_soft_threshold(beta_ols, lam)`：对每个系数做 $\text{sign}(\beta)\cdot\max(|\beta|-\lambda,0)$，
用 `numpy` 向量化实现（不要写 Python for 循环）。

In [ ]:
def lasso_soft_threshold(beta_ols, lam):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
beta = np.array([3.0, -0.5, 1.2, 0.05, -2.0])
out = lasso_soft_threshold(beta, 1.0)
assert np.allclose(out, [2.0, 0.0, 0.2, 0.0, -1.0])
assert np.allclose(lasso_soft_threshold(beta, 0.0), beta)     # lambda=0 时等于不惩罚
assert np.all(np.abs(lasso_soft_threshold(beta, 10.0)) < 1e-12)  # lambda 大到能把所有系数清零

print('lam=1.0:', out)
print('\n✅ 练习 1 通过。')

## ✏️ 练习 2：有效样本数权重（呼应 C58-01）

实现 `effective_number_weight(counts, beta=0.999)`：按 Cui et al. 2019 的「有效样本数」公式
$E_n=(1-\beta^n)/(1-\beta)$，返回权重 $w_c \propto 1/E_{n_c}$，**归一化到权重之和等于类别数**
（即平均权重为 1）。

In [ ]:
def effective_number_weight(counts, beta=0.999):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
w = effective_number_weight([10, 100, 1000], beta=0.99)
assert len(w) == 3
assert abs(sum(w) - 3.0) < 1e-9                 # 归一化到 sum=类别数
assert w[0] > w[1] > w[2]                       # 样本越少权重越大
assert abs(w[0] - 2.406841855617004) < 1e-6
assert abs(w[2] - 0.2301471597560314) < 1e-6

print('counts=[10,100,1000] 的权重:', [round(x, 4) for x in w])
print('\n✅ 练习 2 通过：稀有类样本数越少，权重越大——这是类别不平衡重加权的骨架公式。')

## ✏️ 练习 3：Walk-forward 切分生成器

实现 `walk_forward_splits(n, n_splits=5, min_train_frac=0.5)`：训练集从 0 开始、长度不断增长，
测试块紧跟其后、大小相等（最后一块吸收余数）。返回 `[(train_idx_list, test_idx_list), ...]`。

In [ ]:
def walk_forward_splits(n, n_splits=5, min_train_frac=0.5):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
sp = walk_forward_splits(100, n_splits=5, min_train_frac=0.5)
assert len(sp) == 5
for tr, te in sp:
    assert max(tr) < min(te)                    # 训练永远早于验证，不会看到"未来"
assert sp[0][0] == list(range(50))              # 第一折训练集是前50个
assert sp[0][1] == list(range(50, 60))          # 第一折验证集是接下来10个
assert sp[-1][1] == list(range(90, 100))        # 最后一折验证集吸收到数据末尾
assert len(sp[0][0]) < len(sp[-1][0])           # 训练集随折数递增

for tr, te in sp:
    print(len(tr), '训练 ->', len(te), '验证，验证区间', te[0], '~', te[-1])
print('\n✅ 练习 3 通过：这就是第 5 节 CV 泄漏实验里"正确做法"背后的切分逻辑。')

## ✏️ 练习 4：从预测矩阵直接算偏差-方差

实现 `bias_variance_from_preds(preds, y_true)`：`preds` 形状 `(R, T)`（R个模型在T个测试点上的预测），
`y_true` 形状 `(T,)`。返回 `(bias2, var)`。

In [ ]:
def bias_variance_from_preds(preds, y_true):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
preds = np.array([[1., 2.], [3., 4.], [5., 6.]])
y_true = np.array([3., 4.])
b2, v = bias_variance_from_preds(preds, y_true)
assert abs(b2 - 0.0) < 1e-12                    # 均值预测正好等于真值 -> bias为0
assert abs(v - 8/3) < 1e-9                      # 每列方差 (4+0+4)/3 = 8/3

perfect = np.tile(y_true, (5, 1))               # 5个模型全部预测完全正确、且完全一致
b2p, vp = bias_variance_from_preds(perfect, y_true)
assert abs(b2p) < 1e-12 and abs(vp) < 1e-12     # 零偏差零方差

print(f'合成用例: bias2={b2:.4f}, var={v:.4f}')
print('\n✅ 练习 4 通过：本节前面 6 个实验背后用的都是这个函数的等价逻辑。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def lasso_soft_threshold(beta_ols, lam):
    beta_ols = np.asarray(beta_ols, dtype=float)
    return np.sign(beta_ols) * np.maximum(np.abs(beta_ols) - lam, 0.0)

In [ ]:
# 练习 2 参考答案
def effective_number_weight(counts, beta=0.999):
    en = [(1 - beta**n) / (1 - beta) for n in counts]
    w = [1.0 / e for e in en]
    total = sum(w)
    k = len(w)
    return [wi * k / total for wi in w]

In [ ]:
# 练习 3 参考答案
def walk_forward_splits(n, n_splits=5, min_train_frac=0.5):
    start = int(n * min_train_frac)
    remaining = n - start
    step = remaining // n_splits
    splits = []
    for i in range(n_splits):
        train_end = start + i * step
        test_end = start + (i + 1) * step if i < n_splits - 1 else n
        splits.append((list(range(0, train_end)), list(range(train_end, test_end))))
    return splits

In [ ]:
# 练习 4 参考答案
def bias_variance_from_preds(preds, y_true):
    preds = np.asarray(preds, dtype=float)
    y_true = np.asarray(y_true, dtype=float)
    mean_pred = preds.mean(axis=0)
    bias2 = np.mean((mean_pred - y_true) ** 2)
    var = np.mean(preds.var(axis=0))
    return bias2, var

---
## 🧪 真实工程胶囊：九个主题的 60 秒速记卡

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════════
# 每条一句话定义 + 一条最容易被追问到的失效边界（面试当天可以直接照抄）
# ══════════════════════════════════════════════════════════════════════
# 1) 偏差-方差   定义: 泛化误差=Bias^2+Var+噪声
#              失效: 深度网络的双下降——过参数化后误差先骤增后再降，打破"越复杂越容易过拟合"
# 2) 正则化家族  定义: L1/L2/dropout/early-stop/数据增强 效果类似但机制完全不同
#              失效: L1在共线特征上选择不稳定；dropout的"权重缩放"只在线性readout下精确成立
# 3) 交叉验证    定义: 用"装作没见过的数据"估计泛化能力
#              失效: 时间序列/分组数据上标准K折会因数据不可交换而系统性乐观
# 4) 类别不平衡  定义: 重采样/重加权/阈值调整/两级架构，四把刀切在流水线不同阶段
#              完整方案见 C58-01，本课只给问答骨架
# 5) 集成方法    定义: bagging平均降方差，boosting接力改错降偏差
#              失效: bagging基学习器相关性高(如都用同一批数据)时，方差降不动
# 6) 生成vs判别  定义: 生成式建模P(x,y)，判别式直接建模P(y|x)
#              失效: x高维时生成式对P(x)建模不准，会拖累分类精度
# 7) 维度灾难    定义: 高维空间中几乎所有点对距离趋同，"近邻"失去意义
#              失效: 流形假设成立时(数据实际落在低维流形上)，距离度量在流形内依然可靠
# 8) 特征工程/阈值 定义: 训练完之后还能调的旋钮——特征筛选和阈值都不需要重新训练
#              踩雷: 把"调阈值"和"重新训练"混为一谈
#
# ══════════════════════════════════════════════════════════════════════
# 与本课程其他部分的分工
# ══════════════════════════════════════════════════════════════════════
# · 以上九个主题的完整数学推导                    -> C07
# · 类别不平衡的完整工程方案(有效样本数/EQL/解耦训练) -> C58-01
# · 优化器/学习率调度/初始化/混合精度              -> C64 模块 02
# · 卷积/归一化/attention/CNN vs Transformer      -> C64 模块 03
'''
print(RECIPE)
for token in ['双下降', 'dropout', 'C58-01', '流形假设', 'C07']:
    assert token in RECIPE, token
print('✅ 检查单覆盖：九个主题的定义+失效边界速记 / 课程分工')

### 小结

- **偏差-方差分解的经典 U 型有一个重要例外**：深度学习的双下降——过参数化跨过插值阈值后，
  测试误差先骤增再重新下降，这打破"复杂度越高越容易过拟合"的朴素直觉（本 notebook 用随机傅里叶特征
  + 最小范数解完整复现了这个现象）。
- **L1/L2/dropout/early stopping/数据增强是五种不同机制**，效果都指向"抗过拟合"但原理各不相同：
  L1 靠约束区域的尖角产生精确 0，L2 只收缩不清零；dropout 的"权重缩放"等价性**只在线性 readout 下精确成立**，
  一旦有非线性就只是近似（Jensen gap 不随采样数消失）。
- **交叉验证的无偏性依赖"可交换性"假设**：时间序列和分组数据都会打破这个假设，标准 K 折会让
  "未来"或"同组"信息泄漏进训练集，本 notebook 实测泄漏可以让 MSE 被低估一个数量级以上。
- **bagging 降方差、boosting 降偏差**——这不是一句空话，本 notebook 在同一个玩具问题上分别实测了
  两者的偏差-方差变化曲线，数字上验证了这条经典结论。
- 类别不平衡本节**只给问答骨架**，完整方案见 **C58-01**；九个主题的数学推导全部见 **C07**。

下一站：**模块 02 · 优化与训练问答** —— 优化器怎么选、weight decay 和 L2 为什么在 Adam 下不等价、
warmup 为什么必要。